In [168]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
import csv

In [169]:
load_dotenv()
base_url = "https://api.insee.fr/api-sirene/3.11/siret/"
sirene_api_key = os.environ.get("SIRENE_API_KEY")

In [170]:
# open codes NAF
naf_codes = []
with open("output/interesting_naf_codes.csv", "r", encoding="utf-8") as file:
    csv_reader = csv.DictReader(file, delimiter="|")
    for line in csv_reader:
        naf_codes.append(line["Code NAF"][:2] + "." + line["Code NAF"][2:])

In [171]:
print(naf_codes)

['35.11Z', '35.12Z', '35.13Z', '35.14Z', '35.21Z', '35.22Z', '35.23Z', '36.00Z', '37.00Z', '38.11Z', '38.12Z', '38.21Z', '38.22Z', '38.32Z', '39.00Z', '32.11Z', '32.12Z', '32.13Z', '32.20Z', '32.30Z', '32.40Z', '32.50A', '32.50B', '32.91Z', '32.99Z', '33.11Z', '33.12Z', '33.13Z', '33.14Z', '33.15Z', '33.16Z', '33.17Z', '33.19Z', '33.20A', '33.20B', '33.20C', '33.20D', '24.10Z', '24.20Z', '24.31Z', '24.32Z', '24.33Z', '24.34Z', '24.42Z', '24.42Z', '24.43Z', '24.44Z', '24.45Z', '24.46Z', '24.51Z', '24.52Z', '24.53Z', '24.54Z', '25.11Z', '25.12Z', '25.21Z', '25.29Z', '25.30Z', '25.40Z', '25.50A', '25.50B', '25.61Z', '25.62A', '25.62B', '25.71Z', '25.72Z', '25.73A', '25.73B', '25.91Z', '25.92Z', '25.93Z', '25.94Z', '25.99A', '25.99B', '21.10Z', '21.20Z', '28.11Z', '28.12Z', '28.13Z', '28.14Z', '28.15Z', '28.21Z', '28.22Z', '28.23Z', '28.24Z', '28.25Z', '28.29A', '28.29B', '28.30Z', '28.41Z', '28.49Z', '28.91Z', '28.92Z', '28.93Z', '28.94Z', '28.95Z', '28.96Z', '28.99A', '28.99B', '20.11Z',

In [172]:
naf_codes = pd.read_csv("output/interesting_naf_codes.csv", delimiter='|')['Code NAF']

In [173]:
naf_codes = naf_codes.apply(lambda x: x[:2] + "." + x[2:]).unique()

In [174]:
headers = {
    "X-INSEE-Api-Key-Integration": sirene_api_key,
    "Accept-Encoding": "gzip",
    "Accept": "application/json",
}

In [175]:
departement = "69"

In [176]:
from tqdm import tqdm

champs = "siret,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,codeCommuneEtablissement,coordonneeLambertAbscisseEtablissement"

for code in tqdm(naf_codes):
    query = f"activitePrincipaleUniteLegale:{code} AND trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:{departement}*"
    # Parameters for the query
    params = {
        "q": query,
        "champs": champs,
        "nombre": "1000"
    }
    response = requests.get(
        url=base_url,
        headers=headers,
        params=params
    )
    try:
        if response.json()["header"]["total"] > 1000:
            print(f"plus de 1000 entrées sur : {query}")
    except Exception as e:
        print(e)
        print(response.json())
    break

  0%|          | 0/276 [00:00<?, ?it/s]


In [177]:
from pandas.io.json._normalize import json_normalize

In [178]:
etablissements = json_normalize(response.json()['etablissements'])

In [179]:
response.json()["header"]

{'statut': 200, 'message': 'OK', 'total': 390, 'debut': 0, 'nombre': 390}

In [180]:
etablissements.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 5 columns):
 #   Column                                                       Non-Null Count  Dtype 
---  ------                                                       --------------  ----- 
 0   siret                                                        390 non-null    object
 1   uniteLegale.activitePrincipaleUniteLegale                    390 non-null    object
 2   uniteLegale.trancheEffectifsUniteLegale                      390 non-null    object
 3   adresseEtablissement.codeCommuneEtablissement                390 non-null    object
 4   adresseEtablissement.coordonneeLambertAbscisseEtablissement  324 non-null    object
dtypes: object(5)
memory usage: 15.4+ KB


# slicing

In [181]:
from tqdm import tqdm

champs = "siret,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,codeCommuneEtablissement,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement"

taille_slice = 20

etablissements = pd.DataFrame()

for i in tqdm(range(0, len(naf_codes), taille_slice)):
    naf_groupe = naf_codes[i:min(i+taille_slice, len(naf_codes)-1)]
    query = f"trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:{departement}* AND (activitePrincipaleUniteLegale:{naf_groupe[0]} "
    for code in naf_groupe[1:]:
        query += f"OR activitePrincipaleUniteLegale:{code} "
    query += ")"
    # Parameters for the query
    params = {
        "q": query,
        "champs": champs,
        "nombre": "1000"
    }
    response = requests.get(
        url=base_url,
        headers=headers,
        params=params
    )
    try:
        if response.json()["header"]["total"] > 1000:
            print(f"plus de 1000 entrées sur : {query}")
        # executer le traitement normal
        etablissements = pd.concat([etablissements, json_normalize(response.json()['etablissements'])])
    except Exception as e:
        print(e)
        print(response.json())

  7%|▋         | 1/14 [00:00<00:05,  2.32it/s]

plus de 1000 entrées sur : trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:69* AND (activitePrincipaleUniteLegale:35.11Z OR activitePrincipaleUniteLegale:35.12Z OR activitePrincipaleUniteLegale:35.13Z OR activitePrincipaleUniteLegale:35.14Z OR activitePrincipaleUniteLegale:35.21Z OR activitePrincipaleUniteLegale:35.22Z OR activitePrincipaleUniteLegale:35.23Z OR activitePrincipaleUniteLegale:36.00Z OR activitePrincipaleUniteLegale:37.00Z OR activitePrincipaleUniteLegale:38.11Z OR activitePrincipaleUniteLegale:38.12Z OR activitePrincipaleUniteLegale:38.21Z OR activitePrincipaleUniteLegale:38.22Z OR activitePrincipaleUniteLegale:38.32Z OR activitePrincipaleUniteLegale:39.00Z OR activitePrincipaleUniteLegale:32.11Z OR activitePrincipaleUniteLegale:32.12Z OR activitePrincipaleUniteLegale:32.13Z OR activitePrincipaleUniteLegale:32.20Z OR activitePrincipaleUniteLegale:32.30Z )


 14%|█▍        | 2/14 [00:00<00:04,  2.45it/s]

plus de 1000 entrées sur : trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:69* AND (activitePrincipaleUniteLegale:32.40Z OR activitePrincipaleUniteLegale:32.50A OR activitePrincipaleUniteLegale:32.50B OR activitePrincipaleUniteLegale:32.91Z OR activitePrincipaleUniteLegale:32.99Z OR activitePrincipaleUniteLegale:33.11Z OR activitePrincipaleUniteLegale:33.12Z OR activitePrincipaleUniteLegale:33.13Z OR activitePrincipaleUniteLegale:33.14Z OR activitePrincipaleUniteLegale:33.15Z OR activitePrincipaleUniteLegale:33.16Z OR activitePrincipaleUniteLegale:33.17Z OR activitePrincipaleUniteLegale:33.19Z OR activitePrincipaleUniteLegale:33.20A OR activitePrincipaleUniteLegale:33.20B OR activitePrincipaleUniteLegale:33.20C OR activitePrincipaleUniteLegale:33.20D OR activitePrincipaleUniteLegale:24.10Z OR activitePrincipaleUniteLegale:24.20Z OR activitePrincipaleUniteLegale:24.31Z )


 29%|██▊       | 4/14 [00:01<00:03,  2.64it/s]

plus de 1000 entrées sur : trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:69* AND (activitePrincipaleUniteLegale:25.61Z OR activitePrincipaleUniteLegale:25.62A OR activitePrincipaleUniteLegale:25.62B OR activitePrincipaleUniteLegale:25.71Z OR activitePrincipaleUniteLegale:25.72Z OR activitePrincipaleUniteLegale:25.73A OR activitePrincipaleUniteLegale:25.73B OR activitePrincipaleUniteLegale:25.91Z OR activitePrincipaleUniteLegale:25.92Z OR activitePrincipaleUniteLegale:25.93Z OR activitePrincipaleUniteLegale:25.94Z OR activitePrincipaleUniteLegale:25.99A OR activitePrincipaleUniteLegale:25.99B OR activitePrincipaleUniteLegale:21.10Z OR activitePrincipaleUniteLegale:21.20Z OR activitePrincipaleUniteLegale:28.11Z OR activitePrincipaleUniteLegale:28.12Z OR activitePrincipaleUniteLegale:28.13Z OR activitePrincipaleUniteLegale:28.14Z OR activitePrincipaleUniteLegale:28.15Z )


 50%|█████     | 7/14 [00:02<00:02,  3.03it/s]

plus de 1000 entrées sur : trancheEffectifsUniteLegale:[0 TO 53] AND codeCommuneEtablissement:69* AND (activitePrincipaleUniteLegale:10.31Z OR activitePrincipaleUniteLegale:10.32Z OR activitePrincipaleUniteLegale:10.39A OR activitePrincipaleUniteLegale:10.39B OR activitePrincipaleUniteLegale:10.41A OR activitePrincipaleUniteLegale:10.41B OR activitePrincipaleUniteLegale:10.42Z OR activitePrincipaleUniteLegale:10.51A OR activitePrincipaleUniteLegale:10.51B OR activitePrincipaleUniteLegale:10.51C OR activitePrincipaleUniteLegale:10.51D OR activitePrincipaleUniteLegale:10.52Z OR activitePrincipaleUniteLegale:10.61A OR activitePrincipaleUniteLegale:10.61B OR activitePrincipaleUniteLegale:10.62Z OR activitePrincipaleUniteLegale:10.71A OR activitePrincipaleUniteLegale:10.71B OR activitePrincipaleUniteLegale:10.71C OR activitePrincipaleUniteLegale:10.71D OR activitePrincipaleUniteLegale:10.72Z )


100%|██████████| 14/14 [00:04<00:00,  3.34it/s]


In [182]:
# etablissements = json_normalize(response.json()['etablissements'])

In [183]:
etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"].value_counts()

adresseEtablissement.coordonneeLambertAbscisseEtablissement
844817.0525406802    31
844819.6011970437    27
845941.6264788181    22
839901.777246777     22
843213.6001334677    16
                     ..
838132.0962019961     1
850742.0627815668     1
850865.8412998805     1
823152.6776361581     1
861590.7410396768     1
Name: count, Length: 5245, dtype: int64

In [184]:
etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"].value_counts()

adresseEtablissement.coordonneeLambertOrdonneeEtablissement
6514649.258048379     31
6515036.1056378875    27
6525526.726296        22
6520621.374708257     22
6515718.024287252     16
                      ..
6523911.862731171      1
6519417.9547724845     1
6521492.386886951      1
6525858.718417043      1
6513400.395949277      1
Name: count, Length: 5245, dtype: int64

In [185]:
etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"] = pd.to_numeric(etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"], errors='coerce')
etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"] = pd.to_numeric(etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"], errors='coerce')

In [186]:
etablissements.dropna(inplace=True)

In [187]:
# verify that we are able to plot 
import geopandas as gpd

In [188]:
geo_etablissement = gpd.GeoDataFrame(
    etablissements,
    geometry=gpd.points_from_xy(etablissements["adresseEtablissement.coordonneeLambertAbscisseEtablissement"], etablissements["adresseEtablissement.coordonneeLambertOrdonneeEtablissement"]),
    crs="EPSG:9794"
    )

In [189]:
geo_etablissement = geo_etablissement.to_crs("EPSG:3857")

In [190]:
geo_etablissement['x'] = geo_etablissement.geometry.x
geo_etablissement['y'] = geo_etablissement.geometry.y

In [191]:
from bokeh.io import output_file, output_notebook, show
from bokeh.plotting import figure, ColumnDataSource
output_notebook()

Loading BokehJS ...

In [192]:
# Step 2: Convert to ColumnDataSource for Bokeh
source = ColumnDataSource(data=dict(
    x=geo_etablissement['x'],
    y=geo_etablissement['y'],
    # add more columns here if you want tooltips or coloring based on other attributes
))

In [193]:
# Step 3: Create the Bokeh plot
p = figure(title="Geospatial Points", x_axis_type="mercator", y_axis_type="mercator")
p.add_tile("OSM")  # Add OpenStreetMap tiles (or other tile provider)

# Plot the points
p.circle(x='x', y='y', source=source, size=5, color="blue", alpha=0.7)

# Display the plot
show(p)